In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'notebook_connected'
eu = pd.read_csv('/Users/qingyidrechsler/wrds-rd-work/eu236/outputs/final_all_country_20260928.csv', low_memory=False, parse_dates=['position_date'])

In [2]:
# coverage by country
cov = (eu.groupby('country')
         .agg(disclosures=('position_date', 'size'), companies=('isin', 'nunique'),
              holders=('position_holder', 'nunique'),
              first=('position_date', 'min'), last=('position_date', 'max'))
         .sort_values('disclosures', ascending=False))
total = pd.DataFrame({'disclosures': [len(eu)], 'companies': [eu['isin'].nunique()],
                      'holders': [eu.position_holder.nunique()],
                      'first': [eu.position_date.min()], 'last': [eu.position_date.max()]},
                     index=['All countries'])
cov = pd.concat([cov, total])
out = pd.DataFrame({
    'Disclosures': cov.disclosures.map('{:,}'.format),
    'Companies (ISINs)': cov.companies.map('{:,}'.format),
    'Position holders': cov.holders.map('{:,}'.format),
    'First position date': cov['first'].dt.strftime('%Y-%m-%d'),
    'Last position date': cov['last'].dt.strftime('%Y-%m-%d'),
}, index=cov.index.rename('Country'))
out

,Disclosures,Companies (ISINs),Position holders,First position date,Last position date
Country,,,,,
UK,"112,510",941,621,2012-10-31,2026-07-09
Belgium,"81,689",63,192,2012-10-31,2026-09-16
Germany,"61,498","1,038",489,2003-03-26,2026-09-21
France,"47,941",271,530,2012-10-03,2026-09-18
Sweden,"39,732",465,486,2010-05-10,2026-09-21
Italy,"25,705",232,307,2012-08-30,2026-09-21
Netherlands,"24,705",99,281,2012-11-01,2026-09-21
Spain,"13,743",90,254,2010-06-10,2026-09-18
Finland,"10,939",80,239,2012-10-31,2026-09-21


In [3]:
# companies with a disclosed position each month: top 4 countries + the rest
eu['month'] = eu.position_date.dt.to_period('M').dt.to_timestamp()
# Nov 2012 (regulation in force) to Jun 2026 (UK and Denmark files end in July)
m = eu.loc[(eu.month >= '2012-11-01') & (eu.month <= '2026-06-01')]
by = m.drop_duplicates(['country', 'month', 'isin']).groupby(['month', 'country']).size()
by = by.unstack(fill_value=0)
top = by.sum().sort_values(ascending=False).index[:4]
series = [(c, by[c], color) for c, color in
          zip(top, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100'])]
series.append(('All other countries', by.drop(columns=top).sum(axis=1), '#8a8984'))

fig = go.Figure()
for name, s, color in series:
    fig.add_trace(go.Scatter(x=s.index, y=s, name=name, mode='lines',
                             line=dict(color=color, width=2),
                             hovertemplate='%{y:,}<extra>' + name + '</extra>'))
fig.update_layout(
    title=dict(text='Companies with a Disclosed Net Short Position, by Month'
                    '<br><sup style="color:#52514e">Distinct ISINs with at least one '
                    'position disclosure dated in the month, '
                    f'{by.index[0]:%b %Y}\u2013{by.index[-1]:%b %Y}</sup>',
               x=0, xanchor='left'),
    template='plotly_white', height=500, margin=dict(l=60, r=30, t=90, b=110),
    font=dict(family='Inter, system-ui, sans-serif', size=13, color='#0b0b0b'),
    paper_bgcolor='#fcfcfb', plot_bgcolor='#fcfcfb', hovermode='x unified',
    legend=dict(orientation='h', x=0, y=-0.12, yanchor='top'),
    xaxis=dict(showgrid=False, ticks='outside', tickcolor='#c3c2b7',
               linecolor='#c3c2b7', hoverformat='%b %Y'),
    yaxis=dict(title='Companies', gridcolor='#ecebe7', zeroline=False, rangemode='tozero'),
)
fig.show()